# Análisis de filtrado de full text
Compara qué papers pasan o no el `FullTextFilter` (Step 6).

In [112]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.paper import Paper
from utils.intermediate_io import STEP4_FILE, STEP5_FILE, STEP6_FILE, load_model_list

In [113]:
papers_step4 = load_model_list(STEP4_FILE, Paper)
papers_step5 = load_model_list(STEP5_FILE, Paper)
papers_step6 = load_model_list(STEP6_FILE, Paper)

print(f"Step4 (raw full text): {len(papers_step4)}")
print(f"Step5 (clean text): {len(papers_step5)}")
print(f"Step6 (passed filter): {len(papers_step6)}")

Step4 (raw full text): 52
Step5 (clean text): 52
Step6 (passed filter): 49


In [114]:
def pid(p: Paper) -> str:
    return (p.doi or p.title).strip().lower()

def to_df(papers, step_name: str):
    return pd.DataFrame([{
        "paper_id": pid(p),
        "title": p.title,
        "doi": p.doi,
        "format": p.full_text.format.value if p.full_text else None,
        "chars": len(p.full_text.content) if (p.full_text and p.full_text.content) else 0,
        "source": p.source,
        "ft_retrieved_by": p.ft_retrieved_by,
        "step": step_name,
    } for p in papers])

df4 = to_df(papers_step4, "step4")
df5 = to_df(papers_step5, "step5")
df6 = to_df(papers_step6, "step6")

transitions = [
    ("step4", df4, "step5", df5),
    ("step5", df5, "step6", df6),
]

dropped_parts = []
for from_step, from_df, to_step, to_df_ in transitions:
    to_ids = set(to_df_["paper_id"])
    dropped = from_df[~from_df["paper_id"].isin(to_ids)].copy()
    dropped["from_step"] = from_step
    dropped["to_step"] = to_step
    dropped_parts.append(dropped)

dropped_all = pd.concat(dropped_parts, ignore_index=True)
dropped_all.head()

,paper_id,title,doi,format,chars,source,ft_retrieved_by,step,from_step,to_step
0,10.1007/3-540-29832-0_1225,Penicillin-binding Protein,10.1007/3-540-29832-0_1225,html,3474,crossref,semantic_scholar,step5,step5,step6
1,10.1107/s0108767383000586,International tables for X-ray crystallography,10.1107/s0108767383000586,pdf,9363,semantic_scholar,openalex,step5,step5,step6
2,10.1016/j.micpath.2025.107691,Decoding virulence and resistance in Klebsiell...,10.1016/j.micpath.2025.107691,plain,230047,elsevier,elsevier,step5,step5,step6


In [115]:
summary = (
    dropped_all.groupby(["from_step", "to_step", "format"]).agg(
        n=("paper_id", "count"),
        median_chars=("chars", "median"),
    )
    .reset_index()
    .sort_values(["from_step", "n"], ascending=[True, False])
)
summary

,from_step,to_step,format,n,median_chars
0,step5,step6,html,1,3474.0
1,step5,step6,pdf,1,9363.0
2,step5,step6,plain,1,230047.0


In [116]:
cols = ["from_step", "to_step", "paper_id", "title", "doi", "format", "chars", "source", "ft_retrieved_by"]
dropped_all[cols].sort_values(["from_step", "to_step", "chars"]).head(50)


,from_step,to_step,paper_id,title,doi,format,chars,source,ft_retrieved_by
0,step5,step6,10.1007/3-540-29832-0_1225,Penicillin-binding Protein,10.1007/3-540-29832-0_1225,html,3474,crossref,semantic_scholar
1,step5,step6,10.1107/s0108767383000586,International tables for X-ray crystallography,10.1107/s0108767383000586,pdf,9363,semantic_scholar,openalex
2,step5,step6,10.1016/j.micpath.2025.107691,Decoding virulence and resistance in Klebsiell...,10.1016/j.micpath.2025.107691,plain,230047,elsevier,elsevier


## Notas
- Compara pérdidas entre pasos consecutivos: `step4 -> step5` y `step5 -> step6`.
- `dropped_all` contiene el detalle de papers que NO pasan entre un paso y otro.
- La salida impresa agrupa por transición para facilitar revisión rápida en terminal/notebook.